## Transorm Refunds data
### 1. Extract specific portion of the string from refund_reason using split funtion
### 2. extract speific portion of the string from refund_reason using regexp_extract funtion
### 3. Extract date and time from the refund_timestamp
### 4.Write transormed data to tthe sivler schema

In [0]:
df= spark.read.table('gizmobox_catalog_noori.bronze.py1_refunds')
display(df)

In [0]:
from pyspark.sql.functions import split
df.select('refund_id',
        "payment_id",
        "refund_timestamp",
        "refund_amount",
        split("refund_reason", ":")[0].alias("reason"),
        split("refund_reason", ":")[1].alias("reason_source")
        ).display()

In [0]:
from pyspark.sql.functions import regexp_extract
df.select('refund_id',
        "payment_id",
        "refund_timestamp",
        "refund_amount",
        regexp_extract("refund_reason", "^([^:]+):",1).alias("reason"),
        regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("reason_source")
        ).display()

In [0]:
%sql
select 
*,
regexp_extract(refund_reason,'^([^:]+):',1) as reason ,
regexp_extract(refund_reason,'^[^:]+:(.*)$',1) as reason_source
 from practice_sql_db_connection_catalog.dbo.refunds

In [0]:
from pyspark.sql.functions import regexp_extract,cast,date_format
df_formatted = df.select('refund_id',
        "payment_id",
        date_format("refund_timestamp","yyyy-MM-dd").cast("date").alias("refund_date"),
        date_format("refund_timestamp","HH:mm:ss").cast("timestamp").alias("refund_time"),
        "refund_amount",
        regexp_extract("refund_reason", "^([^:]+):",1).alias("reason"),
        regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("reason_source")
        )
display(df_formatted)

In [0]:
df_formatted.writeTo("gizmobox_catalog_noori.silver.py1_refunds").createOrReplace()

In [0]:
spark.read.table("gizmobox_catalog_noori.silver.py1_refunds").display()